# Use API for analysis rather than downloading as storage
- https://www.tycho.pitt.edu/dataset/api/

In [37]:
import pandas as pd

nis_vacc_coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
nis_vacc_coverage_df['year'].min(), nis_vacc_coverage_df['year'].max()

(np.int64(1995), np.int64(2024))

In [38]:
import os
from dotenv import load_dotenv

# Load variables from .env file into the environment
load_dotenv()

# Access the variables
API_KEY = os.getenv("TYCHO_API_KEY")

### Columns needed from this query:
- ConditionName
- ConditionSNOMED
- CountryCode
- Admin1ISO
- Admin1Name
- PeriodStartDate
- PeriodEndDate
- CountValue
- PartOfCumulativeCountSeries
- Fatalities

In [39]:
import pandas as pd

diseases = ["Measles", "Mumps", "Pertussis"]

core_columns = [
    "ConditionName",
    "CountValue",
    "PartOfCumulativeCountSeries",
    "PeriodEndDate",
    "Admin1ISO"
]

territories = ["PR", "VI", "GU", "AS", "MP"]

disease_dfs = []

for disease in diseases:

    print(f"\nDownloading {disease}...")

    pages = []
    offset = 0
    limit = 5000

    while True:

        url = (
            "https://www.tycho.pitt.edu/api/query?"
            f"apikey={API_KEY}"
            f"&ConditionName={disease}"
            "&CountryISO=US"
            "&PeriodStartDate%3E=1995-01-01"
            "&PeriodEndDate%3C=2024-12-31"
            "&Fatalities=0"
            f"&limit={limit}"
            f"&offset={offset}"
        )

        temp_df = pd.read_csv(url)

        # Stop when the API returns no usable data
        if not set(core_columns).issubset(temp_df.columns):
            break

        temp_df = temp_df[core_columns].copy()
        pages.append(temp_df)

        print(
            f"{disease}: downloaded {len(temp_df):,} rows "
            f"| offset: {offset:,}"
        )

        if len(temp_df) < limit:
            break

        offset += limit

    if not pages:
        print(f"{disease}: no valid rows returned")
        continue

    # Combine pages for this disease
    disease_df = pd.concat(
        pages,
        ignore_index=True
    )

    # Convert date and numeric fields
    disease_df["PeriodEndDate"] = pd.to_datetime(
        disease_df["PeriodEndDate"],
        errors="coerce"
    )

    disease_df["CountValue"] = pd.to_numeric(
        disease_df["CountValue"],
        errors="coerce"
    )

    disease_df["PartOfCumulativeCountSeries"] = pd.to_numeric(
        disease_df["PartOfCumulativeCountSeries"],
        errors="coerce"
    )

    # Derive year from the parsed date
    disease_df["year"] = disease_df["PeriodEndDate"].dt.year

    # Extract the state abbreviation
    disease_df["state"] = disease_df["Admin1ISO"].str.extract(
        r"^US-([A-Z]{2})$",
        expand=False
    )

    # Remove unusable observations
    disease_df = disease_df.dropna(
        subset=[
            "ConditionName",
            "CountValue",
            "PartOfCumulativeCountSeries",
            "PeriodEndDate",
            "year",
            "state"
        ]
    ).copy()

    disease_df["year"] = disease_df["year"].astype(int)

    disease_df["PartOfCumulativeCountSeries"] = (
        disease_df["PartOfCumulativeCountSeries"]
        .astype(int)
    )

    # Exclude US territories
    disease_df = disease_df[
        ~disease_df["state"].isin(territories)
    ].copy()

    # Admin1ISO is no longer needed; PeriodEndDate is retained
    disease_df.drop(
        columns=["Admin1ISO"],
        inplace=True
    )

    print(f"{disease} final: {disease_df.shape}")

    disease_dfs.append(disease_df)


# Combine all three diseases
df = pd.concat(
    disease_dfs,
    ignore_index=True
)

print("\nCombined:", df.shape)

df.head()


Measles: downloaded 5,000 rows | offset: 0
Measles: downloaded 5,000 rows | offset: 5,000
Measles: downloaded 3,495 rows | offset: 10,000
Measles final: (13182, 6)

Mumps: downloaded 5,000 rows | offset: 0
Mumps: downloaded 5,000 rows | offset: 5,000
Mumps: downloaded 5,000 rows | offset: 10,000
Mumps: downloaded 5,000 rows | offset: 15,000
Mumps: downloaded 5,000 rows | offset: 20,000
Mumps: downloaded 563 rows | offset: 25,000
Mumps final: (24883, 6)

Pertussis: downloaded 5,000 rows | offset: 0
Pertussis: downloaded 5,000 rows | offset: 5,000
Pertussis: downloaded 5,000 rows | offset: 10,000
Pertussis: downloaded 5,000 rows | offset: 15,000
Pertussis: downloaded 5,000 rows | offset: 20,000
Pertussis: downloaded 5,000 rows | offset: 25,000
Pertussis: downloaded 5,000 rows | offset: 30,000
Pertussis: downloaded 5,000 rows | offset: 35,000
Pertussis: downloaded 5,000 rows | offset: 40,000
Pertussis: downloaded 5,000 rows | offset: 45,000
Pertussis: downloaded 5,000 rows | offset: 50,0

,ConditionName,CountValue,PartOfCumulativeCountSeries,PeriodEndDate,year,state
0,Measles,1,0,1995-05-27,1995,OH
1,Measles,1,0,1995-10-14,1995,OH
2,Measles,2,0,1996-03-30,1996,OH
3,Measles,3,0,1996-09-14,1996,OH
4,Measles,1,0,1996-12-14,1996,OH


In [40]:
df.groupby("ConditionName").agg(
    first_date=("PeriodEndDate", "min"),
    last_date=("PeriodEndDate", "max"),
    first_year=("year", "min"),
    last_year=("year", "max"),
    rows=("CountValue", "size")
)

,first_date,last_date,first_year,last_year,rows
ConditionName,,,,,
Measles,1995-01-07,2002-12-28,1995,2002,13182
Mumps,1995-01-07,2017-12-30,1995,2017,24883
Pertussis,1995-01-07,2017-12-30,1995,2017,70648


In [41]:
measles_after_2002 = df[
    (df["ConditionName"] == "Measles") &
    (df["year"] > 2002)
]

print(measles_after_2002.shape)

(0, 6)


In [42]:
df.info(), df[df["state"].isin(territories)]

<class 'pandas.DataFrame'>
RangeIndex: 108713 entries, 0 to 108712
Data columns (total 6 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   ConditionName                108713 non-null  str           
 1   CountValue                   108713 non-null  int64         
 2   PartOfCumulativeCountSeries  108713 non-null  int64         
 3   PeriodEndDate                108713 non-null  datetime64[us]
 4   year                         108713 non-null  int64         
 5   state                        108713 non-null  str           
dtypes: datetime64[us](1), int64(3), str(2)
memory usage: 6.0 MB


(None,
 Empty DataFrame
 Columns: [ConditionName, CountValue, PartOfCumulativeCountSeries, PeriodEndDate, year, state]
 Index: [])

In [43]:
assert False

AssertionError: 

### Aggregate for non-cumulative records

In [ ]:
group_columns = [
    "year",
    "state",
    "ConditionName"
]

In [ ]:
non_cumulative_df = (
    df[df["PartOfCumulativeCountSeries"] == 0]
    .groupby(group_columns, as_index=False)
    .agg(
        cases=("CountValue", "sum"),
        records=("CountValue", "size")
    )
)

non_cumulative_df["reporting_type"] = "non_cumulative"
non_cumulative_df["final_report_date"] = pd.NaT

### Aggregate for cumulative records

In [ ]:
cumulative_rows = df[
    df["PartOfCumulativeCountSeries"] == 1
].copy()

latest_dates = (
    cumulative_rows
    .groupby(group_columns)["PeriodEndDate"]
    .transform("max")
)

cumulative_final_rows = cumulative_rows[
    cumulative_rows["PeriodEndDate"] == latest_dates
].copy()

In [ ]:
cumulative_df = (
    cumulative_final_rows
    .groupby(group_columns, as_index=False)
    .agg(
        cases=("CountValue", "max"),
        final_report_date=("PeriodEndDate", "max"),
        final_date_records=("CountValue", "size")
    )
)

cumulative_df["reporting_type"] = "cumulative"

### Combine both non-cumulative and cumulative

In [ ]:
state_year_cases_long = pd.concat(
    [cumulative_df, non_cumulative_df],
    ignore_index=True
)

state_year_cases_long["priority"] = (
    state_year_cases_long["reporting_type"]
    .map({
        "cumulative": 1,
        "non_cumulative": 2
    })
)

state_year_cases_long = (
    state_year_cases_long
    .sort_values("priority")
    .drop_duplicates(
        subset=group_columns,
        keep="first"
    )
    .drop(columns="priority")
    .sort_values(group_columns)
    .reset_index(drop=True)
)

state_year_cases_long.head()

,year,state,ConditionName,cases,final_report_date,final_date_records,reporting_type,records
0,1995,AK,Mumps,13,1995-12-30,1.0,cumulative,NaN
1,1995,AK,Pertussis,1,1995-12-30,1.0,cumulative,NaN
2,1995,AL,Mumps,4,1995-12-30,1.0,cumulative,NaN
3,1995,AL,Pertussis,38,1995-12-30,1.0,cumulative,NaN
4,1995,AR,Measles,2,1995-12-30,2.0,cumulative,NaN


### QC

In [ ]:
cumulative_df[
    cumulative_df["final_date_records"] > 1
].sort_values(
    "final_date_records",
    ascending=False
).head(20)

,year,state,ConditionName,cases,final_report_date,final_date_records,reporting_type
16,1995,CT,Measles,2,1995-12-30,3,cumulative
10,1995,CA,Measles,108,1995-12-30,3,cumulative
45,1995,LA,Measles,18,1995-12-30,3,cumulative
48,1995,MA,Measles,5,1995-12-30,3,cumulative
33,1995,ID,Measles,2,1995-12-30,3,cumulative
36,1995,IL,Measles,6,1995-12-30,3,cumulative
218,1996,TX,Measles,28,1996-12-28,3,cumulative
221,1996,UT,Measles,119,1996-12-28,3,cumulative
205,1996,OR,Measles,11,1996-12-28,3,cumulative
200,1996,OH,Measles,6,1996-12-28,3,cumulative


In [ ]:
state_year_cases_long.loc[
    (state_year_cases_long["year"] == 1998) &
    (state_year_cases_long["ConditionName"] == "Measles"),
    "cases"
].sum()

np.int64(92)

In [ ]:
final_date_check = (
    cumulative_final_rows
    .groupby(
        ["year", "state", "ConditionName", "PeriodEndDate"],
        as_index=False
    )
    .agg(
        final_date_records=("CountValue", "size"),
        unique_values=("CountValue", "nunique"),
        minimum_value=("CountValue", "min"),
        maximum_value=("CountValue", "max")
    )
)

final_date_check[
    final_date_check["final_date_records"] > 1
].head(20)

,year,state,ConditionName,PeriodEndDate,final_date_records,unique_values,minimum_value,maximum_value
4,1995,AR,Measles,1995-12-30,2,1,2,2
7,1995,AZ,Measles,1995-12-30,2,1,10,10
10,1995,CA,Measles,1995-12-30,3,3,3,108
13,1995,CO,Measles,1995-12-30,2,1,24,24
16,1995,CT,Measles,1995-12-30,3,2,1,2
21,1995,FL,Measles,1995-12-30,2,1,9,9
24,1995,GA,Measles,1995-12-30,2,1,2,2
27,1995,HI,Measles,1995-12-30,2,1,1,1
33,1995,ID,Measles,1995-12-30,3,2,1,2
36,1995,IL,Measles,1995-12-30,3,3,2,6


In [ ]:
final_date_disagreements = final_date_check[
    final_date_check["unique_values"] > 1
].sort_values(
    ["year", "state", "ConditionName"]
)

final_date_disagreements

,year,state,ConditionName,PeriodEndDate,final_date_records,unique_values,minimum_value,maximum_value
10,1995,CA,Measles,1995-12-30,3,3,3,108
16,1995,CT,Measles,1995-12-30,3,2,1,2
33,1995,ID,Measles,1995-12-30,3,2,1,2
36,1995,IL,Measles,1995-12-30,3,3,2,6
45,1995,LA,Measles,1995-12-30,3,3,1,18
...,...,...,...,...,...,...,...,...
766,2001,MA,Measles,2001-12-29,3,3,1,3
768,2001,MD,Measles,2001-12-29,3,3,1,3
774,2001,MN,Measles,2001-12-29,3,3,1,3
804,2001,PA,Measles,2001-12-29,3,3,1,6


In [ ]:
maximum_method = (
    cumulative_rows
    .groupby(group_columns, as_index=False)
    .agg(
        maximum_cases=("CountValue", "max")
    )
)

latest_method = cumulative_df[
    group_columns + ["cases", "final_report_date"]
].copy()

comparison = latest_method.merge(
    maximum_method,
    on=group_columns,
    how="left"
)

comparison["difference"] = (
    comparison["cases"] -
    comparison["maximum_cases"]
)

comparison[
    (comparison["year"] == 1998) &
    (comparison["ConditionName"] == "Measles") &
    (comparison["difference"] != 0)
]

,year,state,ConditionName,cases,final_report_date,maximum_cases,difference
364,1998,AL,Measles,1,1998-12-26,2,-1


### Quantify revisions accross entire dataset

In [ ]:
revisions = comparison[
    comparison["difference"] != 0
].copy()

print(f"Revised state-disease-years: {len(revisions):,}")

revisions["difference"].value_counts().sort_index()

Revised state-disease-years: 332


difference
-4211     1
-2931     1
-1784     1
-1694     1
-1039     1
         ..
-5        8
-4       11
-3       17
-2       24
-1       57
Name: count, Length: 130, dtype: int64

In [ ]:
revision_summary = (
    revisions
    .groupby("ConditionName", as_index=False)
    .agg(
        revised_state_years=("difference", "size"),
        total_net_revision=("difference", "sum"),
        largest_downward_revision=("difference", "min"),
        largest_upward_revision=("difference", "max")
    )
)

revision_summary

,ConditionName,revised_state_years,total_net_revision,largest_downward_revision,largest_upward_revision
0,Measles,25,-71,-16,-1
1,Mumps,91,-1207,-533,-1
2,Pertussis,216,-35522,-4211,-1


In [ ]:
cases_df = (
    state_year_cases_long
    .pivot(
        index=["year", "state"],
        columns="ConditionName",
        values="cases"
    )
    .reset_index()
)

cases_df.columns.name = None

cases_df.rename(
    columns={
        "Measles": "measles_cases",
        "Mumps": "mumps_cases",
        "Pertussis": "pertussis_cases"
    },
    inplace=True
)

cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


# lets reformat this table

In [ ]:
cases_df.to_csv('../app/data/tycho_cases.csv', index=False)

In [ ]:
assert False

AssertionError: 

In [ ]:
import pandas as pd

tp_1 = pd.read_table('../misc/genecounts1.txt')
tp_2 = pd.read_table('../misc/genecounts2.txt')

tps_df = pd.merge(tp_1, tp_2, on=' gene')


tps_df.rename(columns={'0_x': 'tp_1', '0_y': 'tp_2', ' gene': 'gene'}, inplace=True)
tps_df = tps_df.dropna()
tps_df

,tp_1,gene,tp_2
6,1079,dnaA,1079
7,35,dnaN,4
8,35,dnaN,0
9,0,yaaA,0
10,3983,recF,0
...,...,...,...
2217,0,yidC,0
2218,0,yidC,1
2219,0,rnpA,0
2220,0,rpmH,0


In [ ]:
tps_df['diff'] = abs(tps_df['tp_1'] - tps_df['tp_2'])
tps_df

,tp_1,gene,tp_2,diff
10,3983,recF,0,3983
6,1079,dnaA,1079,0
1911,880,ackA,588,292
234,524,cysS,134,390
1603,372,ftsA,488,116
...,...,...,...,...
1106,0,flgB,0,0
1105,0,flgK,0,0
1104,0,cheR,0,0
1103,0,fliY,0,0


In [ ]:
tps_df = tps_df[(tps_df['tp_1'] > 0) & (tps_df['tp_2'] > 0)]

In [ ]:
tps_df = tps_df.sort_values(by='diff', ascending=False)
tps_df.head(10)

,tp_1,gene,tp_2,diff
12,52,gyrA,7832,7780
234,524,cysS,134,390
1911,880,ackA,588,292
940,30,prpE,249,219
11,215,gyrB,22,193
1831,80,folC,262,182
229,201,disA,51,150
1261,23,spoVS,153,130
228,166,radA,42,124
753,125,cadA,3,122


In [ ]:
import plotly.express as px

plot_df = tps_df.head(10).melt(
    id_vars="gene",
    value_vars=["tp_1", "tp_2"],
    var_name="time_point",
    value_name="counts"
)

fig = px.bar(
    plot_df,
    x="gene",
    y="counts",
    color="time_point",
    barmode="group",
    text="counts",
    title="Top 10 Genes by Count",
    labels={
        "gene": "Gene",
        "counts": "Count",
        "time_point": "Time Point",
    },
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Gene",
    yaxis_title="Count",
    legend_title="Time Point",
)

fig.show()